# U.S. Treasury 10-Year Yield Curve Analysis Pipeline
## End-to-End Databricks Medallion Architecture (PySpark & SQL)

### Executive Overview
This project implements a multi-stage **Medallion Data Lakehouse Architecture** on Databricks to ingest, cleanse, and model daily U.S. Treasury 10-Year Yield Curve data. It combines Python-based REST API extraction with advanced Databricks SQL transformations, window functions, and time-series analytics tailored for downstream reporting and BI integration.

---

### Architecture & Data Flow

```text
  [ U.S. Treasury API ] 
            │
            ▼ (Python / Requests / PySpark)
   ┌─────────────────┐
   │  BRONZE LAYER   │  Raw string array ingestion & audit metadata appending
   │ (bronze_nasdaq) │
   └────────┬────────┘
            │
            ▼ (Databricks SQL / Regex CTE / Quality Filters)
   ┌─────────────────┐
   │  SILVER LAYER   │  Schema enforcement, array tokenization, date coercion,
   │ (silver_nasdaq) │  and holiday 'N/A' value cleansing
   └────────┬────────┘
            │
            ▼ (Databricks SQL / Window Functions)
   ┌─────────────────┐
   │   GOLD LAYER    │  Time-series analytics: Day-over-day deltas (LAG), 
   │  (gold_nasdaq)  │  30-day sliding moving average (AVG), & yearly ranks (DENSE_RANK)
   └─────────────────┘

In [0]:
%python
#01 - Bronze Layer - Extract/Ingestion

import requests
import csv
from io import StringIO
from pyspark.sql.functions import current_timestamp, lit

#Define API endpoint, data.nasdaq.com blocks non-browser/datacenter requests with a 403 bot challenge, so we pull the same U.S. Treasury daily yield curve data from the official Treasury CSV feed
url = "https://home.treasury.gov/resource-center/data-chart-center/interest-rates/daily-treasury-rates.csv/2026/all?type=daily_treasury_yield_curve&field_tdr_date_value=2026&page&_format=csv"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

#API requests
response = requests.get(url, headers=headers)
response.raise_for_status()

#Parse CSV response into rows (skip header); keep the 100 most recent rows, which are listed newest-first
rows = list(csv.reader(StringIO(response.text)))[1:101]

#Turn string arrays into Spark Dataframe
raw_json_df = spark.createDataFrame([(str(row),) for row in rows], schema=["raw_payload"])

#Add metadata columns
bronze_df = (raw_json_df.withColumn("ingested_at", current_timestamp()).withColumn("source_url",lit(url)))

#Persist from memory to storage with Delta Lake format table
(bronze_df.write.format("delta").mode("append").saveAsTable("bronze_nasdaq_api"))

print(f"Successfully ingested {bronze_df.count()} raw records into bronze_nasdaq_api.")

Successfully ingested 100 raw records into bronze_nasdaq_api.


In [0]:
%sql
--02 - Silver Layer - Transform

CREATE OR REPLACE TABLE silver_nasdaq_yields AS 

-- CTE 1: Parse raw string payload into an indexted array for column extraction
WITH parsed_payload AS (
    -- CGet rid of brackets, quotes and split into a SQL array
    SELECT split(regexp_replace(raw_payload, '[\\[\\]\\\']', ''), ',\\s*') AS payload_arr
    FROM bronze_nasdaq_api
)

SELECT
    -- Index 0: Trade Date (handles MM/dd/yyyy and yyyy-MM-dd formats)
    coalesce(
        to_date(payload_arr[0], 'MM/dd/yyyy'),
        to_date(payload_arr[0], 'yyyy-MM-dd')
    ) AS trade_date,

    -- Index 11: 10 Year Treasury Yield Rate
    CAST(payload_arr[11] AS DOUBLE) AS rate_10yr
FROM parsed_payload

-- Data Quality Layer: Filter out empty rows, nulls, and market holiday 'N/A' strings
WHERE payload_arr[0] IS NOT NULL
AND payload_arr[11] IS NOT NULL
AND payload_arr[11] !='N/A';

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- 03 - Gold Layer - Load

CREATE OR REPLACE TABLE gold_nasdaq_yield_analytics AS

-- CTE 1: Calculate time-series window metrics and day over day averages
WITH base_window_metrics AS (
    SELECT trade_date,
    rate_10yr,
    
    -- WINDOW Function 1: Navigation/Relative Offset: LAG accesses the 10 year yield from 1 row prior ordered chronologically
    LAG(rate_10yr, 1) OVER (ORDER BY trade_date ASC
    ) AS prior_day_rate,

    -- Day over day absolute yield difference using LAG
    rate_10yr - LAG(rate_10yr, 1 ) OVER (
        ORDER BY trade_date ASC
    ) AS rate_change_dod,

    -- WINDOW Function 2: Sliding Aggregation Frame: Computes a 30 day trailing moving average using an explicit physical frame, current row + 29 preceeding rows
    AVG(rate_10yr) OVER (
        ORDER BY trade_date ASC
        ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
    ) AS rate_30day_ma
    FROM silver_nasdaq_yields
),

-- CTE 2: Partition and rank yearly interest rate highs
ranked_yearly_spikes AS (
    SELECT 
    trade_date,
    rate_10yr,
    prior_day_rate,
    ROUND(rate_change_dod, 4) AS rate_change_dod,
    ROUND(rate_30day_ma, 4) AS rate_30day_mavg,

    -- WINDOW Function 3: PARTITION BY resets the ranking logic at the boundary of each calendar year.
        -- DENSE_RANK assigns consecutive rank positions (1, 2, 2, 3) without skipping rank numbers when rate values tie.
        DENSE_RANK() OVER (
            PARTITION BY YEAR(trade_date)
            ORDER BY rate_10yr DESC
        ) AS yearly_yield_rank
        FROM base_window_metrics
)

-- Final Selection: Executive reporting layer formatted for downstream analytics / Power BI
SELECT 
    trade_date,
    rate_10yr,
    prior_day_rate,
    rate_change_dod,
    rate_30day_mavg,
    yearly_yield_rank
FROM ranked_yearly_spikes
ORDER BY trade_date DESC;

num_affected_rows,num_inserted_rows
